In [1]:
import json
import numpy as np
from skimage.draw import polygon
from tqdm import tqdm
import os
################ CHANGE THIS TO YOUR LOCAL FOLDER #############
openslide_path = r'C:\Users\labadmin\Documents\openslide-win64-20230414\bin'
###############################################################
os.environ['PATH'] = openslide_path + ";" + os.environ['PATH']
# from openslide import OpenSlide
if hasattr(os, 'add_dll_directory'):
    # Python >= 3.8 on Windows
    with os.add_dll_directory(openslide_path):
        import openslide
else:
    import openslide
from openslide import OpenSlide
from tifffile import imwrite

def create_and_save_mask(json_dir, wsi_dir, output_path, target_resolution=2.0):
    """
    Converts all JSON files with cell polygons in a directory into a single mask with the specified resolution.

    Args:
    - json_dir (str): Path to the directory containing JSON files.
    - wsi_dir (str): Path to the directory containing WSI files.
    - output_path (str): Path to save the output mask as a TIFF file.
    - target_resolution (float): Desired resolution in µm/pixel.

    Returns:
    - None
    """
    mask = None
    
    json_files = sorted([os.path.join(json_dir, f) for f in os.listdir(json_dir) if f.endswith('.json')])
    
    cell_label = 1
    for json_file in json_files:
        # Get corresponding WSI file path
        json_filename = os.path.basename(json_file)
        wsi_filename = json_filename.replace('.json', '.ndpi')  # Assuming TIFF format for WSIs
        wsi_path = os.path.join(wsi_dir, wsi_filename)
        
        # Get image dimensions and resolution from WSI
        with OpenSlide(wsi_path) as slide:
            width, height = slide.dimensions
            x_res = float(slide.properties.get('openslide.mpp-x', 1.0))  # Resolution in µm/pixel
            y_res = float(slide.properties.get('openslide.mpp-y', 1.0))  # Resolution in µm/pixel

        # Calculate new dimensions based on target resolution
        scale_x = target_resolution/x_res
        scale_y = target_resolution/y_res
        scaled_width = int(width / scale_x)
        scaled_height = int(height / scale_y)
        
        # Create a new mask if necessary
        if mask is None or mask.shape != (scaled_height, scaled_width):
            mask = np.zeros((scaled_height, scaled_width), dtype=np.uint32)
        
        # Load polygons from JSON
        with open(json_file, 'r') as f:
            cells = json.load(f)
        
        for cell in tqdm(cells, desc=f"Percentage of cells in {json_filename}"):
            # Extract contours (32 contours per cell)
            contours = [np.array(contour) for contour in cell['contour']]
            
            for contour in contours:
                # Scale contour
                contour[:, 0] = contour[:, 0] / scale_x
                contour[:, 1] = contour[:, 1] / scale_y
                
                # Create mask
                rr, cc = polygon(contour[:, 1], contour[:, 0], (scaled_height, scaled_width))
                mask[rr, cc] = cell_label
                
            cell_label += 1
            
    # Save the mask as a compressed TIFF file
    imwrite(output_path, mask, compress=6)


In [1]:
import os
import numpy as np
import tensorflow as tf
import gc
from skimage.io import imread
from tifffile import imwrite

def segment_and_save_JSONs_and_masks(WSIs, model, out_pth_json, out_pth_tif=None) -> None:
    """
    Segments whole slide images (WSIs) and saves the results as GeoJSON files and TIFF masks.

    Args:
    - WSIs (list): List of paths to the whole slide images.
    - model (object): Trained model for predicting instances.
    - out_pth_json (str): Path to save the output JSON files.
    - out_pth_tif (str, optional): Path to save optional TIFF files. Default is None.

    Returns:
    - None
    """
    total_images = len(WSIs)
    
    for idx, img_pth in enumerate(WSIs, start=1):
        try:
            name = os.path.basename(img_pth)
            nm = os.path.splitext(os.path.basename(name))[0]
            json_output_path = os.path.join(out_pth_json, nm + '.json')

            if not os.path.exists(json_output_path):
                print(f'\nStarting {nm} ({idx}/{total_images})')

                # Read and normalize image
                img = imread(img_pth)
                img = img / 255  # normalization used to train model

                # Predict instances
                mask, polys = model.predict_instances_big(
                    img,
                    axes='YXC',
                    block_size=4096,
                    min_overlap=128,
                    context=128,
                    n_tiles=(4, 4, 1)
                )

                # Save JSON results
                # print('Saving JSON...')
                # save_json_from_WSI_pred(polys, out_pth_json, name)

                # Save mask
                # if out_pth_tif:
                print('Saving TIFF mask...')
                tif_output_path = os.path.join(out_pth_tif, nm + '.tif')
                imwrite(tif_output_path, mask)

                # Clear GPU memory
                tf.keras.backend.clear_session()
                gc.collect()
            else:
                print(f'Skipping {nm} ({idx}/{total_images})')
        except Exception as e:
            print(f'Skipping {img_pth} ({idx}/{total_images}), probably because it\'s too big... Error: {e}')
            # Clear GPU memory in case of exception
            tf.keras.backend.clear_session()
            gc.collect()


In [2]:
import matplotlib.pyplot as plt
import numpy as np
from stardist.models import StarDist2D, Config2D
import json
import geojson
from typing import List, Dict, Tuple
from pathlib import Path
import os
from tifffile import imread, imwrite
from tqdm import tqdm
import random
from PIL import Image
from stardist import fill_label_holes
import copy
from tensorflow.python.summary.summary_iterator import summary_iterator
import struct
from matplotlib.colors import ListedColormap
import pandas as pd
import gc
import tensorflow as tf
pth_WSI_ndpi = r'\\10.162.80.16\Andre\data\Ashleigh fallopian tube\fallopian tubes\AJER376'
date = '12_11_23'
json_dir =os.path.join(pth_WSI_ndpi, f'StarDist_{date}', 'json')
output_path = os.path.join(pth_WSI_ndpi, f'StarDist_{date}', 'stardist_masks')
os.makedirs(output_path, exist_ok=True)
def load_model(model_path: str) -> StarDist2D:
    """Justin made: Load StarDist model weights, configurations, and thresholds"""
    # TODO: remove offshoot thing
    with open(model_path + '\\config.json', 'r') as f:
        config = json.load(f)
    with open(model_path + '\\thresholds.json', 'r') as f:
        thresh = json.load(f)
    model = StarDist2D(config=Config2D(**config), basedir=model_path, name='offshoot_model')
    model.thresholds = thresh
    print('Overriding defaults:', model.thresholds, '\n')
    model.load_weights(model_path + '\\weights_best.h5')
    return model
def load_selected_models_AF(model_name)-> StarDist2D:
    base_path = r'\\10.162.80.16\Andre\data\Stardist\models'
    model_paths = {
        "pdac": os.path.join(base_path, "lea_model"),
        "panin": os.path.join(base_path, "Big_PANIN_model_8_10_24_lr_0.001_epochs_400_pt_40"),
        "monkey": os.path.join(base_path, "monkey_12_12_2023_lr_0.001_epochs_400_pt_10_gaus_ratio_0"),
        "fallopian_tube": os.path.join(base_path, "fallopian_tube_12_7_2023_lr_0.001_epochs_400_pt_40")
    }

    if model_name in model_paths:
        model = load_model(model_paths[model_name])
        print(f"Loaded {model_name} model successfully.")
        return model
    else:
        raise ValueError("Invalid model name provided. Choose from 'lea_model', 'panin', 'monkey', or 'fallopian tube'.")

model_name = "monkey"  #"pdac" "panin"  "monkey"  "fallopian_tube"  
model = load_selected_models_AF(model_name)

segment_and_save_JSONs_and_masks(pth_WSI_ndpi, model, json_dir, output_path)

# create_and_save_mask(json_dir, pth_WSI_ndpi, output_path, target_resolution=2.0)


base_model.py (149): output path for model already exists, files may be overwritten: \\10.162.80.16\andre\data\Stardist\models\monkey_12_12_2023_lr_0.001_epochs_400_pt_10_gaus_ratio_0\offshoot_model


Using default values: prob_thresh=0.5, nms_thresh=0.4.
Overriding defaults: Thresholds(prob=0.6883699162882626, nms=0.3) 

Loaded monkey model successfully.

Starting  (1/73)
Skipping \ (1/73), probably because it's too big... Error: [Errno 2] No such file or directory: 'C:\\'

Starting  (2/73)
Skipping \ (2/73), probably because it's too big... Error: [Errno 2] No such file or directory: 'C:\\'

Starting 1 (3/73)
Skipping 1 (3/73), probably because it's too big... Error: [Errno 2] No such file or directory: 'C:\\Users\\labadmin\\PycharmProjects\\CODA_HE_nucleus_segmentation\\workflow\\1'

Starting 0 (4/73)
Skipping 0 (4/73), probably because it's too big... Error: [Errno 2] No such file or directory: 'C:\\Users\\labadmin\\PycharmProjects\\CODA_HE_nucleus_segmentation\\workflow\\0'

Starting . (5/73)
Skipping . (5/73), probably because it's too big... Error: [Errno 13] Permission denied: 'C:\\Users\\labadmin\\PycharmProjects\\CODA_HE_nucleus_segmentation\\workflow'

Starting 1 (6/73)
S

KeyboardInterrupt: 